### Load Environment Variables

Load API keys and configuration from a `.env` file in the project root. This allows secure storage of sensitive credentials like OpenAI API keys without hardcoding them in the notebook.

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

### Simple Agent Example

Create a basic LangChain agent with two simple tools (`search` and `get_weather`) and demonstrate how the agent can use these tools to answer questions. The agent uses the ReAct (Reasoning + Acting) pattern to decide when to use tools and how to combine their results.

### Create a File System Agent with ReAct Pattern

Define a comprehensive set of file system tools (read, write, delete, rename, list directory, create directory, check file existence) and create a ReAct agent that can use these tools to perform file operations. The agent can reason about which tools to use and execute complex multi-step file operations.

### Example: Create and Verify a File

Demonstrate the agent's ability to perform a multi-step task: create a file with specific content, then read it back to verify. This shows how the agent can chain tool calls together to accomplish complex goals.

### Complex Example: Directory Management

Demonstrate the agent handling a complex multi-step task involving directory listing, directory creation, file creation with different content types (text and JSON), and verification. This showcases the agent's ability to plan and execute sequential operations.

### Interactive Examples

Template code for experimenting with the file system agent. Uncomment and modify the query to test different file operations. The agent can handle natural language instructions and translate them into appropriate tool calls.

<img src="./agent.svg" alt="Alt text" width="400" height="400">


### Simple Agent Implementation

This code creates a basic LangChain agent with two simple tools (`search` and `get_weather`), then demonstrates it answering a question. The agent automatically decides to use the `get_weather` tool and formats the response.

In [15]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage

model = ChatOpenAI(model="gpt-4o-mini")

@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

agent = create_agent(model, tools=[search, get_weather])


result = agent.invoke(
    {"messages": [HumanMessage("Hello, what is the weather in SF?")]}
)

import pprint
pprint.pprint(result)

{'messages': [HumanMessage(content='Hello, what is the weather in SF?', additional_kwargs={}, response_metadata={}, id='1bca6a3d-d3d5-465c-b327-dfbefcbc7527'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 70, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1590f93f9d', 'id': 'chatcmpl-D3JajqlCIZLlW0rHC6bsacW24OAPR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c094e-7f42-7613-b6cb-d8e8a033ffe4-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'San Francisco'}, 'id': 'call_vA5zC6syrO8q7cPeSMPPhrfj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata=

### File System Tools Implementation

This code defines seven file system tools using the `@tool` decorator and creates a ReAct agent with these tools. Each tool handles file operations (read, write, delete, rename, list, create directory, check existence) with proper error handling.

In [22]:
import os
from pathlib import Path
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# Initialize the model
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# File System Tools

@tool
def read_file(file_path: str) -> str:
    """Read the contents of a file.
    
    Args:
        file_path: The path to the file to read (relative or absolute)
    
    Returns:
        The contents of the file as a string
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: File '{file_path}' not found."
    except PermissionError:
        return f"Error: Permission denied reading '{file_path}'."
    except Exception as e:
        return f"Error reading file: {str(e)}"

@tool
def write_file(file_path: str, content: str) -> str:
    """Write content to a file. Creates the file if it doesn't exist, overwrites if it does.
    
    Args:
        file_path: The path to the file to write (relative or absolute)
        content: The content to write to the file
    
    Returns:
        Success message or error message
    """
    try:
        # Create directory if it doesn't exist
        dir_path = os.path.dirname(file_path)
        if dir_path:
            os.makedirs(dir_path, exist_ok=True)
        
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)
        return f"Successfully wrote {len(content)} characters to '{file_path}'"
    except PermissionError:
        return f"Error: Permission denied writing to '{file_path}'."
    except Exception as e:
        return f"Error writing file: {str(e)}"

@tool
def delete_file(file_path: str) -> str:
    """Delete a file.
    
    Args:
        file_path: The path to the file to delete (relative or absolute)
    
    Returns:
        Success message or error message
    """
    try:
        if os.path.exists(file_path):
            os.remove(file_path)
            return f"Successfully deleted '{file_path}'"
        else:
            return f"Error: File '{file_path}' does not exist."
    except PermissionError:
        return f"Error: Permission denied deleting '{file_path}'."
    except Exception as e:
        return f"Error deleting file: {str(e)}"

@tool
def rename_file(old_path: str, new_path: str) -> str:
    """Rename or move a file.
    
    Args:
        old_path: The current path of the file (relative or absolute)
        new_path: The new path for the file (relative or absolute)
    
    Returns:
        Success message or error message
    """
    try:
        if os.path.exists(old_path):
            os.rename(old_path, new_path)
            return f"Successfully renamed '{old_path}' to '{new_path}'"
        else:
            return f"Error: File '{old_path}' does not exist."
    except PermissionError:
        return f"Error: Permission denied renaming '{old_path}'."
    except Exception as e:
        return f"Error renaming file: {str(e)}"

@tool
def list_directory(directory_path: str = ".") -> str:
    """List files and directories in a directory.
    
    Args:
        directory_path: The path to the directory to list (defaults to current directory)
    
    Returns:
        A formatted string listing files and directories
    """
    try:
        if not os.path.exists(directory_path):
            return f"Error: Directory '{directory_path}' does not exist."
        
        if not os.path.isdir(directory_path):
            return f"Error: '{directory_path}' is not a directory."
        
        items = []
        for item in sorted(os.listdir(directory_path)):
            item_path = os.path.join(directory_path, item)
            if os.path.isdir(item_path):
                items.append(f"[DIR]  {item}/")
            else:
                size = os.path.getsize(item_path)
                items.append(f"[FILE] {item} ({size} bytes)")
        
        if not items:
            return f"Directory '{directory_path}' is empty."
        
        return f"Contents of '{directory_path}':\n" + "\n".join(items)
    except PermissionError:
        return f"Error: Permission denied accessing '{directory_path}'."
    except Exception as e:
        return f"Error listing directory: {str(e)}"

@tool
def create_directory(directory_path: str) -> str:
    """Create a directory (and parent directories if needed).
    
    Args:
        directory_path: The path to the directory to create (relative or absolute)
    
    Returns:
        Success message or error message
    """
    try:
        os.makedirs(directory_path, exist_ok=True)
        return f"Successfully created directory '{directory_path}'"
    except PermissionError:
        return f"Error: Permission denied creating '{directory_path}'."
    except Exception as e:
        return f"Error creating directory: {str(e)}"

@tool
def file_exists(file_path: str) -> str:
    """Check if a file or directory exists.
    
    Args:
        file_path: The path to check (relative or absolute)
    
    Returns:
        Information about whether the file/directory exists
    """
    try:
        if os.path.exists(file_path):
            if os.path.isdir(file_path):
                return f"'{file_path}' exists and is a directory."
            else:
                size = os.path.getsize(file_path)
                return f"'{file_path}' exists and is a file ({size} bytes)."
        else:
            return f"'{file_path}' does not exist."
    except Exception as e:
        return f"Error checking file: {str(e)}"

# Collect all file system tools
file_system_tools = [
    read_file,
    write_file,
    delete_file,
    rename_file,
    list_directory,
    create_directory,
    file_exists
]

# Create the ReAct agent
# Using create_agent which internally uses ReAct pattern
# This is the same API used in cell 2 and works reliably
agent = create_agent(model, tools=file_system_tools)
agent_executor = None  # create_agent returns a runnable, not an executor

print("✓ ReAct File System Agent created successfully!")
print("Note: The agent uses the ReAct (Reasoning + Acting) pattern internally.")

print(f"\nAvailable tools:")
for tool in file_system_tools:
    print(f"  - {tool.name}: {tool.description.split('.')[0]}")

✓ ReAct File System Agent created successfully!
Note: The agent uses the ReAct (Reasoning + Acting) pattern internally.

Available tools:
  - read_file: Read the contents of a file
  - write_file: Write content to a file
  - delete_file: Delete a file
  - rename_file: Rename or move a file
  - list_directory: List files and directories in a directory
  - create_directory: Create a directory (and parent directories if needed)
  - file_exists: Check if a file or directory exists


### Basic File Operation Example

Demonstrate the agent performing a two-step task: creating a file with content and then reading it back to verify. The agent automatically chains the `write_file` and `read_file` tool calls.

In [23]:
# Example: Use the file system agent
from langchain.messages import HumanMessage

result = agent.invoke({
    "messages": [HumanMessage("Create a file called 'test.txt' with the content 'Hello, World!', then read it back to verify.")]
})

print("\n" + "="*50)
print("Agent Response:")
print("="*50)
# Extract the final message content
final_message = result["messages"][-1]
print(final_message.content if hasattr(final_message, 'content') else final_message)


Agent Response:
The file 'test.txt' was successfully created with the content 'Hello, World!'. When I read it back, the content is confirmed as 'Hello, World!'.


### Complex Multi-Step File Operations

Demonstrate the agent handling a complex task with multiple steps: listing directories, creating directories, creating multiple files with different content types, and verifying the results. This shows the agent's planning and execution capabilities.

In [24]:
# More complex example: List directory, create files, and organize them
query = """
1. List the current directory
2. Create a directory called 'data'
3. Create a file 'data/notes.txt' with content 'Important notes'
4. Create a file 'data/config.json' with content '{"version": "1.0"}'
5. List the 'data' directory to verify
"""

result = agent.invoke({
    "messages": [HumanMessage(query)]
})

print("\n" + "="*50)
print("Agent Response:")
print("="*50)
final_message = result["messages"][-1]
print(final_message.content if hasattr(final_message, 'content') else final_message)


Agent Response:
Here are the results of your requests:

1. **Current Directory:**
   ```
   [FILE] agent.svg (12704 bytes)
   [FILE] agents.ipynb (19088 bytes)
   [DIR]  data/
   [FILE] test.txt (13 bytes)
   ```

2. **Created Directory:** Successfully created directory 'data'.

3. **Created Files:**
   - Successfully wrote to 'data/notes.txt' with content: "Important notes".
   - Successfully wrote to 'data/config.json' with content: `{"version": "1.0"}`.

4. **Contents of 'data' Directory:**
   ```
   [FILE] config.json (18 bytes)
   [FILE] notes.txt (15 bytes)
   ``` 

Everything has been set up successfully!


### Experimentation Template

Template code for testing the file system agent with custom queries. Uncomment and modify the query to experiment with different file operations. The agent understands natural language and translates it into appropriate tool calls.

In [ ]:
# Interactive example: Try your own commands!
# Uncomment and modify the command below:

# result = agent.invoke({
#     "messages": [HumanMessage("Your command here - e.g., 'Read the README.md file and summarize it'")]
# })
# print(result["messages"][-1].content)

# Example commands you can try:
# - "Read the requirements.txt file"
# - "Create a backup of README.md called README_backup.md"
# - "List all Python files in the current directory"
# - "Create a new directory 'output' and write a file 'output/result.txt' with content 'Success'"
# - "Check if a file called 'test.txt' exists"
# - "Read README.md, then create a summary file called 'summary.txt'"
# - "List the current directory, then create a file called 'test.txt' with content 'Hello World'"